# AMP Challenge — Step 5: Physicochemical AMP Realism Benchmarking v2

**Pipeline:** Step 4 external near-match scoring → **Step 5 physicochemical benchmarking** → Step 6 potency/activity prediction.

v2 adds column-collision protection, per-sequence error handling, explicit Biopython pI/charge method documentation, and a caution flag for instability index on peptides shorter than 10 aa.


In [1]:
# ============================================================
# AMP Challenge — Step 5
# Physicochemical AMP Realism Benchmarking
# ============================================================
#
# INPUT 1:
#   ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv
#
# INPUT 2:
#   APD6 / ADP6 FASTA reference
#
# PURPOSE:
#   Compute physicochemical descriptors for every retained candidate
#   and compare them against:
#     (A) literature AMP ranges from the review:
#         - length: 10–50 aa
#         - net charge at pH 7: +2 to +9
#         - pI: 3.38–10.17
#     (B) the empirical APD6 natural-AMP distribution
#
# IMPORTANT:
#   - These literature ranges are soft benchmarking criteria, NOT
#     competition hard filters.
#   - Challenge-valid peptides of length 8–9 are retained.
#   - No peptide is removed in this notebook.
#   - The output adds physicochemical features for later
#     multi-objective ranking.
#
# FEATURES:
#   length
#   molecular_weight
#   GRAVY
#   isoelectric_point
#   net_charge_pH7
#   aliphatic_index
#   aromaticity
#   instability_index
#
# APD6 comparison:
#   5th–95th percentile membership for the six core descriptors
#
# OUTPUT:
#   ALL_CLEAN_WITH_NEARMATCH_AND_PHYSCHEM.csv
#   APD6_PHYSCHEM_REFERENCE_SUMMARY.csv
#   STEP5_PHYSCHEM_SUMMARY.csv
#   STEP5_PHYSCHEM_MANIFEST.txt
# ============================================================

!pip -q install biopython pandas numpy tqdm

from google.colab import files
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm
from datetime import datetime
import pandas as pd
import numpy as np
import os
import re

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Literature AMP benchmarking ranges from the uploaded review
LIT_LENGTH_MIN = 10
LIT_LENGTH_MAX = 50
LIT_CHARGE_MIN = 2.0
LIT_CHARGE_MAX = 9.0
LIT_PI_MIN = 3.38
LIT_PI_MAX = 10.17

# Challenge hard-validity range (for audit only)
CHALLENGE_LENGTH_MIN = 8
CHALLENGE_LENGTH_MAX = 50

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_fasta(filename):
    records = []
    current_id = None
    seq_parts = []

    def flush():
        nonlocal current_id, seq_parts
        if current_id is not None:
            seq = "".join(seq_parts).upper().strip()
            if seq:
                records.append((current_id, seq))

    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if line.startswith(">"):
                flush()
                current_id = line[1:].strip() or f"unnamed_{len(records)+1}"
                seq_parts = []
            elif line:
                seq_parts.append(re.sub(r"\s+", "", line))
        flush()

    return records


def aliphatic_index(seq):
    """
    Ikai-style aliphatic index:
    AI = X(Ala) + 2.9*X(Val) + 3.9*(X(Ile)+X(Leu))
    where X = mole percentage.
    """
    L = len(seq)
    if L == 0:
        return np.nan
    pct_A = 100.0 * seq.count("A") / L
    pct_V = 100.0 * seq.count("V") / L
    pct_I = 100.0 * seq.count("I") / L
    pct_L = 100.0 * seq.count("L") / L
    return pct_A + 2.9 * pct_V + 3.9 * (pct_I + pct_L)


def calc_physchem(seq):
    seq = str(seq).upper().strip()

    if not seq:
        return None, "EMPTY_SEQUENCE"

    invalid = set(seq) - STANDARD_AA
    if invalid:
        return None, "INVALID_AA_" + "".join(sorted(invalid))

    try:
        pa = ProteinAnalysis(seq)
        props = {
            "physchem_length": len(seq),
            "molecular_weight": float(pa.molecular_weight()),
            "gravy": float(pa.gravy()),
            "isoelectric_point": float(pa.isoelectric_point()),
            "net_charge_pH7": float(pa.charge_at_pH(7.0)),
            "aliphatic_index": float(aliphatic_index(seq)),
            "aromaticity": float(pa.aromaticity()),
            "instability_index": float(pa.instability_index()),
        }
        return props, None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)[:200]}"


CORE_FEATURES = [
    "physchem_length",
    "molecular_weight",
    "gravy",
    "isoelectric_point",
    "net_charge_pH7",
    "aliphatic_index",
]

# ------------------------------------------------------------
# 1. Upload Step-4 scored clean CSV
# ------------------------------------------------------------

print("\n==============================================")
print("UPLOAD STEP-4 SCORED CLEAN CSV")
print("==============================================")
print("Upload ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv\n")

csv_upload = files.upload()
csv_files = list(csv_upload.keys())

if len(csv_files) != 1:
    raise ValueError("Please upload exactly ONE scored clean CSV.")

input_csv = csv_files[0]
df = pd.read_csv(input_csv)

if "sequence" not in df.columns:
    raise ValueError("Input CSV must contain a column named 'sequence'.")

df["sequence"] = df["sequence"].astype(str).str.upper().str.strip()

print(f"Loaded candidates: {len(df):,}")
print("Input:", input_csv)

# ------------------------------------------------------------
# 2. Upload APD6 / ADP6 reference FASTA
# ------------------------------------------------------------

print("\n==============================================")
print("UPLOAD APD6 / ADP6 REFERENCE FASTA")
print("==============================================")

apd_upload = files.upload()
apd_files = list(apd_upload.keys())

if len(apd_files) != 1:
    raise ValueError("Please upload exactly ONE APD6/ADP6 FASTA file.")

apd_file = apd_files[0]
apd_records = read_fasta(apd_file)

print(f"APD6 records loaded: {len(apd_records):,}")
print("Reference:", apd_file)

# ------------------------------------------------------------
# 3. Calculate APD6 physicochemical distribution
# ------------------------------------------------------------

apd_rows = []

for rid, seq in tqdm(apd_records, desc="Calculating APD6 physicochemical properties"):
    props, err = calc_physchem(seq)
    if props is None:
        continue
    props["reference_id"] = rid
    props["sequence"] = seq
    apd_rows.append(props)

apd_df = pd.DataFrame(apd_rows)

if apd_df.empty:
    raise ValueError("No valid APD6 sequences were available for physicochemical calculation.")

# Empirical reference summary:
# mean, std, median, P05, P95
ref_summary_rows = []

for col in CORE_FEATURES:
    vals = apd_df[col].dropna().astype(float)

    ref_summary_rows.append({
        "feature": col,
        "n": len(vals),
        "mean": vals.mean(),
        "std": vals.std(ddof=1),
        "median": vals.median(),
        "p05": vals.quantile(0.05),
        "p95": vals.quantile(0.95),
        "min": vals.min(),
        "max": vals.max(),
    })

ref_summary_df = pd.DataFrame(ref_summary_rows)

print("\n==============================================")
print("APD6 PHYSICOCHEMICAL REFERENCE DISTRIBUTION")
print("==============================================")
display(ref_summary_df.round(4))

# Lookup dictionary for 5th–95th interval
ref_ranges = {
    row["feature"]: (float(row["p05"]), float(row["p95"]))
    for _, row in ref_summary_df.iterrows()
}

# ------------------------------------------------------------
# 4. Calculate candidate physicochemical descriptors
# ------------------------------------------------------------

# Refuse silent overwriting of physicochemical columns from an earlier run.
PHYS_COLUMNS = [
    "physchem_length", "molecular_weight", "gravy", "isoelectric_point",
    "net_charge_pH7", "aliphatic_index", "aromaticity", "instability_index",
]
collisions = [c for c in PHYS_COLUMNS if c in df.columns]
if collisions:
    raise ValueError(
        "Input CSV already contains physicochemical columns that this step "
        "would overwrite: " + ", ".join(collisions) +
        ". Please use the Step-4 input file or rename/remove those columns explicitly."
    )

candidate_props = []
physchem_errors = []
invalid_count = 0

for seq in tqdm(df["sequence"], total=len(df), desc="Calculating candidate physicochemical features"):
    props, err = calc_physchem(seq)

    if props is None:
        invalid_count += 1
        candidate_props.append({c: np.nan for c in PHYS_COLUMNS})
        physchem_errors.append(err)
    else:
        candidate_props.append(props)
        physchem_errors.append(None)

props_df = pd.DataFrame(candidate_props)

for col in props_df.columns:
    df[col] = props_df[col].values

df["physchem_error"] = physchem_errors

# Instability index is retained as an exploratory descriptor, but for very
# short peptides it is less interpretable because it is dipeptide-based.
df["instability_index_short_peptide_caution"] = df["physchem_length"] < 10

# ------------------------------------------------------------
# 5. Literature AMP-range soft benchmarking
# ------------------------------------------------------------

df["lit_amp_length_10_50"] = (
    (df["physchem_length"] >= LIT_LENGTH_MIN) &
    (df["physchem_length"] <= LIT_LENGTH_MAX)
)

df["lit_amp_charge_2_9"] = (
    (df["net_charge_pH7"] >= LIT_CHARGE_MIN) &
    (df["net_charge_pH7"] <= LIT_CHARGE_MAX)
)

df["lit_amp_pI_3_38_10_17"] = (
    (df["isoelectric_point"] >= LIT_PI_MIN) &
    (df["isoelectric_point"] <= LIT_PI_MAX)
)

df["challenge_length_8_50"] = (
    (df["physchem_length"] >= CHALLENGE_LENGTH_MIN) &
    (df["physchem_length"] <= CHALLENGE_LENGTH_MAX)
)

lit_cols = [
    "lit_amp_length_10_50",
    "lit_amp_charge_2_9",
    "lit_amp_pI_3_38_10_17",
]

df["n_literature_amp_ranges_passed"] = df[lit_cols].sum(axis=1)
df["literature_amp_compliance_pct"] = (
    100.0 * df["n_literature_amp_ranges_passed"] / len(lit_cols)
).round(3)

# ------------------------------------------------------------
# 6. APD6 natural-distribution benchmarking
# ------------------------------------------------------------

apd_pass_cols = []

for feature in CORE_FEATURES:
    p05, p95 = ref_ranges[feature]
    flag_col = f"{feature}_within_APD6_p05_p95"
    apd_pass_cols.append(flag_col)

    df[flag_col] = (
        (df[feature] >= p05) &
        (df[feature] <= p95)
    )

df["n_APD6_core_ranges_passed"] = df[apd_pass_cols].sum(axis=1)
df["APD6_core_distribution_compliance_pct"] = (
    100.0 * df["n_APD6_core_ranges_passed"] / len(apd_pass_cols)
).round(3)

# ------------------------------------------------------------
# 7. Distribution-centered diagnostic score
# ------------------------------------------------------------
# This is NOT an official challenge score.
# For each core property, compute distance from the APD6 median,
# normalized by the APD6 robust 5th–95th interval width.
# Convert to a 0–100 realism score where closer to the APD6 median is better.

diagnostic_component_cols = []

for feature in CORE_FEATURES:
    summary_row = ref_summary_df[ref_summary_df["feature"] == feature].iloc[0]

    med = float(summary_row["median"])
    p05 = float(summary_row["p05"])
    p95 = float(summary_row["p95"])
    width = max(p95 - p05, 1e-9)

    comp_col = f"{feature}_APD6_realism_component"
    diagnostic_component_cols.append(comp_col)

    normalized_distance = (df[feature] - med).abs() / width
    df[comp_col] = (100.0 * (1.0 - normalized_distance.clip(0, 1))).round(3)

df["APD6_physchem_realism_diagnostic"] = (
    df[diagnostic_component_cols].mean(axis=1)
).round(3)

# ------------------------------------------------------------
# 8. Global summary
# ------------------------------------------------------------

summary_rows = [
    {"metric": "input_candidates", "value": len(df)},
    {"metric": "invalid_physchem_sequences", "value": invalid_count},
    {"metric": "literature_length_10_50_pass", "value": int(df["lit_amp_length_10_50"].sum())},
    {"metric": "literature_charge_2_9_pass", "value": int(df["lit_amp_charge_2_9"].sum())},
    {"metric": "literature_pI_3_38_10_17_pass", "value": int(df["lit_amp_pI_3_38_10_17"].sum())},
    {"metric": "all_3_literature_ranges_pass", "value": int((df["n_literature_amp_ranges_passed"] == 3).sum())},
    {"metric": "all_6_APD6_core_ranges_pass", "value": int((df["n_APD6_core_ranges_passed"] == 6).sum())},
]

summary_df = pd.DataFrame(summary_rows)

print("\n")
print("=" * 76)
print("STEP 5 — PHYSICOCHEMICAL AMP REALISM SUMMARY")
print("=" * 76)

display(summary_df)

print("\nCandidate physicochemical distributions:")
display(
    df[CORE_FEATURES + [
        "aromaticity",
        "instability_index",
        "literature_amp_compliance_pct",
        "APD6_core_distribution_compliance_pct",
        "APD6_physchem_realism_diagnostic"
    ]]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
    .T
    .round(4)
)

# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

output_csv = "ALL_CLEAN_WITH_NEARMATCH_AND_PHYSCHEM.csv"
reference_csv = "APD6_PHYSCHEM_REFERENCE_SUMMARY.csv"
summary_csv = "STEP5_PHYSCHEM_SUMMARY.csv"
manifest_txt = "STEP5_PHYSCHEM_MANIFEST.txt"

df.to_csv(output_csv, index=False)
ref_summary_df.to_csv(reference_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

with open(manifest_txt, "w") as f:
    f.write("AMP Challenge — Step 5: Physicochemical AMP Realism Benchmarking\n")
    f.write("=" * 76 + "\n\n")
    f.write(f"Timestamp: {datetime.now().isoformat(timespec='seconds')}\n")
    f.write(f"Input candidate CSV: {input_csv}\n")
    f.write(f"APD6 reference FASTA: {apd_file}\n")
    f.write(f"Candidates: {len(df)}\n")
    f.write(f"Valid APD6 reference sequences: {len(apd_df)}\n\n")

    f.write("Literature AMP soft benchmarking ranges:\n")
    f.write("  length: 10-50 aa\n")
    f.write("  net charge at pH 7: +2 to +9\n")
    f.write("  pI: 3.38-10.17\n\n")

    f.write("Core physicochemical descriptors:\n")
    for x in CORE_FEATURES:
        f.write(f"  - {x}\n")
    f.write("  - aromaticity\n")
    f.write("  - instability_index\n\n")

    f.write("Methodological note:\n")
    f.write(
        "Molecular weight, GRAVY, pI, and charge are calculated with "
        "Bio.SeqUtils.ProtParam.ProteinAnalysis. Therefore pI and charge use "
        "Biopython's ProtParam ionization model/pKa conventions. These may differ "
        "slightly from EMBOSS, Bjellqvist-derived implementations, or the exact "
        "method used by a source paper; this implementation detail is recorded "
        "for reproducibility and literature-range comparisons are treated as soft.\n"
    )
    f.write(
        "The literature AMP ranges are used as soft benchmarking criteria, "
        "not as competition hard filters. Challenge-valid 8-9 residue peptides "
        "are retained even though they fall outside the 10-50 literature AMP "
        "benchmark range.\n"
    )
    f.write(
        "For molecular weight, GRAVY and aliphatic index, no arbitrary fixed "
        "AMP cutoff is introduced. Instead, candidates are compared with the "
        "empirical APD6 natural-AMP distribution using 5th-95th percentile "
        "membership and a distribution-centered diagnostic score.\n"
    )
    f.write(
        "APD6_physchem_realism_diagnostic is a project-specific ranking feature "
        "and is NOT the official AMP Challenge aggregation score.\n"
    )
    f.write(
        "Instability index is retained only as an exploratory descriptor. "
        "For very short peptides (especially 8-9 aa), interpretation is limited "
        "because the index is based on dipeptide composition; such rows are "
        "flagged with instability_index_short_peptide_caution.\n"
    )
    f.write(
        "This notebook is intentionally Colab-oriented because it uses "
        "google.colab.files.upload/download. A separate CLI/local wrapper can "
        "be created later for reproducible repository execution.\n"
    )

print("\nOUTPUT FILES")
print(" -", output_csv)
print(" -", reference_csv)
print(" -", summary_csv)
print(" -", manifest_txt)

# ------------------------------------------------------------
# 10. Download
# ------------------------------------------------------------

for x in [output_csv, reference_csv, summary_csv, manifest_txt]:
    files.download(x)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.7 MB/s eta 0:00:00

UPLOAD STEP-4 SCORED CLEAN CSV
Upload ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv



Saving ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv to ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv
Loaded candidates: 246,795
Input: ALL_CLEAN_WITH_EXTERNAL_NEARMATCH.csv

UPLOAD APD6 / ADP6 REFERENCE FASTA


Saving ADP6.fasta to ADP6.fasta
APD6 records loaded: 2,580
Reference: ADP6.fasta


Calculating APD6 physicochemical properties:   0%|          | 0/2580 [00:00<?, ?it/s]


APD6 PHYSICOCHEMICAL REFERENCE DISTRIBUTION


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# ============================================================
# Step 5B — Soft Multi-Criteria Preselection
# Corrected version
# ============================================================

TARGET_POOL_SIZE = 120000

required_cols = [
    "APD6_physchem_realism_diagnostic",
    "literature_amp_compliance_pct",
    "external_novelty_diagnostic",
    "batch",
    "sequence",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work = df.copy()

# ------------------------------------------------------------
# Safety deduplication
# ------------------------------------------------------------

before_dedup = len(work)

work = work.drop_duplicates(
    subset="sequence",
    keep="first"
).copy()

print("Removed duplicate sequences in Step 5B safety check:",
      before_dedup - len(work))

# ------------------------------------------------------------
# Keep only Challenge-valid rows
# ------------------------------------------------------------

if "challenge_length_8_50" in work.columns:
    work = work[
        work["challenge_length_8_50"] == True
    ].copy()

if len(work) == 0:
    raise ValueError("No candidates remain after validity filtering.")

# ------------------------------------------------------------
# Component scores
# ------------------------------------------------------------

work["score_physchem"] = (
    work["APD6_physchem_realism_diagnostic"]
    .fillna(0)
    .clip(0, 100)
    / 100.0
)

work["score_literature"] = (
    work["literature_amp_compliance_pct"]
    .fillna(0)
    .clip(0, 100)
    / 100.0
)

work["score_novelty"] = (
    work["external_novelty_diagnostic"]
    .fillna(0)
    .clip(0, 100)
    / 100.0
)

# ------------------------------------------------------------
# Temporary preselection score
# NOT the final competition ranking
# ------------------------------------------------------------

work["preselection_score"] = (
    0.45 * work["score_physchem"]
    + 0.25 * work["score_literature"]
    + 0.30 * work["score_novelty"]
)

# Soft penalty only
if "high_similarity_flag_ge80" in work.columns:
    work.loc[
        work["high_similarity_flag_ge80"] == True,
        "preselection_score"
    ] -= 0.10

# ------------------------------------------------------------
# Batch-balanced first-pass selection
# ------------------------------------------------------------

batches = sorted(
    work["batch"]
    .dropna()
    .unique()
)

if len(batches) == 0:
    raise ValueError(
        "No valid batch labels remain after filtering."
    )

if TARGET_POOL_SIZE > len(work):
    print(
        f"Requested {TARGET_POOL_SIZE:,} candidates, "
        f"but only {len(work):,} are available."
    )
    TARGET_POOL_SIZE = len(work)

per_batch_target = TARGET_POOL_SIZE // len(batches)

selected_parts = []

for batch in batches:

    sub = (
        work[work["batch"] == batch]
        .sort_values(
            "preselection_score",
            ascending=False
        )
    )

    selected_parts.append(
        sub.head(per_batch_target)
    )

# IMPORTANT:
# keep original work indices here
preselected = pd.concat(selected_parts)

# ------------------------------------------------------------
# Fill unused slots from globally best remaining candidates
# ------------------------------------------------------------

remaining_slots = TARGET_POOL_SIZE - len(preselected)

if remaining_slots > 0:

    remaining = (
        work.loc[
            ~work.index.isin(preselected.index)
        ]
        .sort_values(
            "preselection_score",
            ascending=False
        )
    )

    preselected = pd.concat([
        preselected,
        remaining.head(remaining_slots)
    ])

# ------------------------------------------------------------
# Final safety checks
# ------------------------------------------------------------

preselected = (
    preselected
    .sort_values(
        "preselection_score",
        ascending=False
    )
    .reset_index(drop=True)
)

# Make sure no duplicate peptide sequence slipped through
duplicate_count = preselected["sequence"].duplicated().sum()

if duplicate_count != 0:
    raise RuntimeError(
        f"Unexpected duplicate sequences in final pool: {duplicate_count}"
    )

if len(preselected) != TARGET_POOL_SIZE:
    raise RuntimeError(
        f"Expected {TARGET_POOL_SIZE:,} candidates, "
        f"but selected {len(preselected):,}."
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n==============================================")
print("STEP 5B — SOFT PRESELECTION SUMMARY")
print("==============================================")

print(f"Input candidates          : {len(df):,}")
print(f"After safety dedup        : {len(work):,}")
print(f"Final preselected pool    : {len(preselected):,}")
print(f"Duplicate sequences final : {duplicate_count}")

print("\nPer-batch representation:")

display(
    preselected["batch"]
    .value_counts()
    .rename_axis("batch")
    .reset_index(name="count")
)

print("\nPreselection score distribution:")

display(
    preselected["preselection_score"]
    .describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    )
    .to_frame("value")
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_preselected = "STEP5B_PRESELECTED_120K.csv"

preselected.to_csv(
    output_preselected,
    index=False
)

print("\nSaved:")
print(output_preselected)

files.download(output_preselected)

Removed duplicate sequences in Step 5B safety check: 0

STEP 5B — SOFT PRESELECTION SUMMARY
Input candidates          : 246,795
After safety dedup        : 246,795
Final preselected pool    : 120,000
Duplicate sequences final : 0

Per-batch representation:


,batch,count
0,hydramp_batch1_seed42,24000
1,hydramp_batch5_seed50,24000
2,hydramp_batch2_seed44,24000
3,hydramp_batch3_seed46,24000
4,hydramp_batch4_seed48,24000



Preselection score distribution:


,value
count,120000.000000
mean,0.708180
std,0.048311
min,0.647211
5%,0.651934
25%,0.669887
50%,0.695424
75%,0.734007
95%,0.807985
max,0.875489



Saved:
STEP5B_PRESELECTED_120K.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>